In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import datasets
from transformers import AutoImageProcessor, AutoModel
import torch.nn as nn
import os
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from tqdm.notebook import tqdm
from icecream import ic
from transformers import get_cosine_schedule_with_warmup
from PIL import Image
import random
import numpy as np
import timm
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)



/home/system/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/system/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias

In [2]:


class OxfordPetDataset(Dataset):
    def __init__(self, root_dir, split="trainval", transform=None):
        """
        root_dir: path to 'oxford-iiit-pet'
        split: 'trainval' or 'test'
        transform: torchvision transforms
        """
        self.root_dir = root_dir
        self.images_dir = os.path.join(root_dir, "images")
        self.annotations_file = os.path.join(root_dir, "annotations", f"{split}.txt")
        # self.transform = transform
        if transform is None:
            self.transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transform

        self.samples = []
        self._load_annotations()
        ic(len(self.samples))

    def _load_annotations(self):
        with open(self.annotations_file, "r") as f:
            for line in tqdm(f):
                parts = line.strip().split()
                image_name = parts[0] + ".jpg"
                label = int(parts[1]) - 1  # convert to 0-based index
                
                self.samples.append((image_name, label))
    
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_name, label = self.samples[idx]

        image_path = os.path.join(self.images_dir, image_name)
        image = Image.open(image_path).convert("RGB")
        image = image.resize((224, 224))

        if self.transform:
            image = self.transform(image)

        return image, label


train_dataset = OxfordPetDataset(root_dir="/media/system/ZERBUIS_EXT_STOR/dynamic_slam/experiments/data/oxford-iiit-pet", split="trainval")
train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=32,      # How many samples per batch
    shuffle=True)

test_dataset = OxfordPetDataset(root_dir="/media/system/ZERBUIS_EXT_STOR/dynamic_slam/experiments/data/oxford-iiit-pet", split="test")
test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=32,      # How many samples per batch
    shuffle=False)


0it [00:00, ?it/s]

ic| len(self.samples): 3680


0it [00:00, ?it/s]

ic| len(self.samples): 3669


In [3]:
for images, labels in train_dataloader:
    print("Batch of images shape:", images.shape)
    print("Batch of labels shape:", labels.shape)
    print(labels)
    break

Batch of images shape: torch.Size([32, 3, 224, 224])
Batch of labels shape: torch.Size([32])
tensor([20, 14, 31,  6, 30, 33, 20, 25, 31,  2, 28,  7,  9, 24, 14, 12, 35, 20,
        22, 24, 32,  3, 10,  9,  1, 13, 11, 33,  9, 12, 11,  0])


In [ ]:
class Classifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
    
        self.dino = AutoModel.from_pretrained('facebook/dinov2-large')
        dino_shape = self.dino.config.hidden_size  # safer

        for param in self.dino.parameters():
            param.requires_grad = False
            
        self.mlp = nn.Sequential(
            nn.Linear(dino_shape, 512),
            nn.ReLU(),
            # nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        outputs = self.dino(pixel_values=x)
        cls_token = outputs.last_hidden_state[:, 0]
        # cls_token = self.temp_network(cls_token)
        logits = self.mlp(cls_token)
        return logits

model_tiny = timm.create_model(
    'vit_tiny_patch16_224',
    pretrained=True,
    num_classes=37
)

class DistillationWrapper(nn.Module):
    def __init__(self, student, teacher_dim=1024):
        super().__init__()
        self.student = student
        self.proj = nn.Linear(student.embed_dim, teacher_dim)

    def forward(self, x):
        student_out = self.student.forward_features(x)
        cls_token = student_out[:, 0]
        projected = self.proj(cls_token)
        return projected
    
    
class Classifier_dist(nn.Module):
    def __init__(self, student, dino_shape=1024, num_classes=37):
        super().__init__()

        self.student = student

        for param in self.student.parameters():
            param.requires_grad = False

        self.mlp = nn.Sequential(
            nn.Linear(dino_shape, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        with torch.no_grad():
            projected = self.student(x)   # ← THIS
        
        logits = self.mlp(projected)
        return logits


# base_model = DistillationWrapper(student=model_tiny)
# base_model.load_state_dict(torch.load("/media/system/ZERBUIS_EXT_STOR/dynamic_slam/experiments/student_weights/student_epoch_12.pt"))
# model = Classifier_dist(student=base_model, num_classes=37)

model = 

model.to("cuda")
print("")

In [5]:
base_model = DistillationWrapper(student=model_tiny)
base_model.load_state_dict(torch.load("/media/system/ZERBUIS_EXT_STOR/dynamic_slam/experiments/student_weights/student_epoch_10.pt"))
model = Classifier_dist(student=base_model, num_classes=37)

model.to("cuda")
print("")

In [6]:
BATCH_SIZE = 32
LR = 5e-5
EPOCHS = 10
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.05

device = "cuda"

model = model.to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

num_training_steps = EPOCHS * len(train_dataloader)
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

criterion = nn.CrossEntropyLoss()

for epoch in tqdm(range(EPOCHS)):

    # ---------------- TRAIN ----------------
    model.train()
    total_loss = 0
    correct, total = 0, 0

    for images, labels in train_dataloader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)           # (B, num_classes)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        # accuracy
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    avg_loss = total_loss / len(train_dataloader)

    # ---------------- VALID ----------------
    model.eval()
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for images, labels in test_dataloader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {avg_loss:.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Val Acc: {val_acc:.2f}%"
    )

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch [1/10] Loss: 2.0667 Train Acc: 56.77% Val Acc: 82.50%
Epoch [2/10] Loss: 0.3382 Train Acc: 90.90% Val Acc: 83.54%
Epoch [3/10] Loss: 0.2379 Train Acc: 92.42% Val Acc: 84.14%
Epoch [4/10] Loss: 0.2008 Train Acc: 93.29% Val Acc: 84.19%
Epoch [5/10] Loss: 0.1855 Train Acc: 93.48% Val Acc: 84.38%
Epoch [6/10] Loss: 0.1739 Train Acc: 93.70% Val Acc: 84.08%
Epoch [7/10] Loss: 0.1625 Train Acc: 94.89% Val Acc: 84.76%
Epoch [8/10] Loss: 0.1562 Train Acc: 94.70% Val Acc: 84.79%
Epoch [9/10] Loss: 0.1526 Train Acc: 94.95% Val Acc: 84.76%
Epoch [10/10] Loss: 0.1521 Train Acc: 94.92% Val Acc: 84.82%
